# tensor-reshape-view — faded example 3: reshape after transpose — fill in the audit dict

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-reshape-view`. The last cell reports your progress on the `PyTorch: reshape vs view` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: reshape vs view` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-reshape-view`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-reshape-view"
DD_SUBTOPIC = "PyTorch: reshape vs view"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After `.transpose()`, `is_contiguous()` returns `False` and `.view()` raises. `.reshape()` always succeeds for the same number of elements, but may allocate a new buffer (copy). You can detect whether a copy occurred by comparing `data_ptr()` before and after: if they differ, a copy was made.

## Faded exercise 3

The report skeleton is provided. **Fill in the four boolean values** in the audit dict.

```python
def audit_reshape(x: t.Tensor) -> dict:
    """x: (M, N) contiguous."""
    y = x.transpose(0, 1)  # (N, M) — likely non-contiguous
    n = x.numel()
    # Attempt view
    view_raised = False
    try:
        _ = y.view(n)
    except RuntimeError:
        view_raised = None  # TODO: set correctly
    r = y.reshape(n)
    return {
        'y_contiguous': None,          # TODO: is y contiguous?
        'view_raised': view_raised,    # already handled above
        'reshape_data_ptr_same': None, # TODO: did reshape share storage?
        'reshape_values_correct': None,# TODO: does r equal y.contiguous().view(n)?
    }
```

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def audit_reshape(x: t.Tensor) -> dict:
    """x: (M, N) contiguous."""
    y = x.transpose(0, 1)   # (N, M) — usually non-contiguous
    n = x.numel()
    view_raised = False
    try:
        _ = y.view(n)
    except RuntimeError:
        view_raised = True
    r = y.reshape(n)
    return {
        'y_contiguous': bool(y.is_contiguous()),
        'view_raised': view_raised,
        'reshape_data_ptr_same': bool(r.data_ptr() == y.data_ptr()),
        'reshape_values_correct': bool(t.allclose(r, y.contiguous().view(n))),
    }

t.manual_seed(2)
print(audit_reshape(t.randn(3, 5)))


def _test():
    import torch as t
    d = audit_reshape(t.randn(4, 3))
    assert d['y_contiguous'] == False, 'transpose should be non-contiguous'
    assert d['view_raised'] == True, 'view should raise on non-contiguous'
    assert d['reshape_data_ptr_same'] == False, 'reshape should copy'
    assert d['reshape_values_correct'] == True, 'values must be correct after copy'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def audit_reshape(x: t.Tensor) -> dict:
    """x: (M, N) contiguous."""
    y = x.transpose(0, 1)   # (N, M) — usually non-contiguous
    n = x.numel()
    view_raised = False
    try:
        _ = y.view(n)
    except RuntimeError:
        view_raised = True
    r = y.reshape(n)
    return {
        'y_contiguous': bool(y.is_contiguous()),
        'view_raised': view_raised,
        'reshape_data_ptr_same': bool(r.data_ptr() == y.data_ptr()),
        'reshape_values_correct': bool(t.allclose(r, y.contiguous().view(n))),
    }

t.manual_seed(2)
print(audit_reshape(t.randn(3, 5)))
```
</details>